# 学习地图

> **本章主题：对象与对象名称（变量）之间的区别。名称不是对象，名称是一个独立的东西。**

白骑士纠结于歌名与歌本身——这正是本章的基调：**名称（变量）只是标签，标签所指的对象才是本体**。

本章按"四个递进的问题"展开：

| 问题 | 对应小节 |
|---|---|
| 变量到底是什么？ | 6.2 变量不是盒子 |
| 两个变量什么时候"是同一个"？ | 6.3 同一性、相等性与别名 |
| 复制对象到底复制了什么？ | 6.4 默认做浅拷贝 |
| 函数会不会改到我传进去的对象？ | 6.5 作为引用的函数参数 |
| 对象什么时候消失？ | 6.6–6.7 del、垃圾回收与不可变类型的把戏 |

> 本章内容相当枯燥，但它是解决 Python 程序中许多不易察觉 bug 的关键。

# 6.2 变量不是盒子

"变量是盒子"是常见但**有害**的比喻，它阻碍了对引用变量的理解。更好的比喻是：**Python 变量是贴在对象上的便利贴（名称标签）**。

In [11]:
# 示例 6.1：变量 a 和 b 保存对同一列表的引用，而不是列表的副本
a = [1, 2, 3]
b = a                # 把标签 b 也贴到 a 所指的同一个对象上
a.append(4)          # 通过 a 修改这个列表
print('b =', b)      # [1, 2, 3, 4] —— 用"盒子"就无法解释
print('b is a =', b is a)

b = [1, 2, 3, 4]
b is a = True


如果 b 是"复制了 a 盒子里的内容"，那么 a 改变后 b 应该纹丝不动。事实并非如此，所以：

> **`b = a` 不会复制对象，它只是给同一个对象再贴一个标签。**

**赋值语句先算右边，再绑定左边。** 对象在绑定之前就已经存在了：

In [2]:
# 示例 6.2：先创建对象，才能绑定变量
class Gizmo:
    def __init__(self):
        print(f'Gizmo id: {id(self)}')

x = Gizmo()             # 创建实例并绑定到 x
try:
    y = Gizmo() * 10    # 右侧先创建实例（打印 id），再做乘法 → 抛 TypeError
except TypeError as e:
    print('TypeError:', e)

# 因为右侧求值时就抛了异常，变量 y 从未被创建
print("'y' in dir() ->", 'y' in dir())

Gizmo id: 1757141180272
Gizmo id: 1757144131792
TypeError: unsupported operand type(s) for *: 'Gizmo' and 'int'
'y' in dir() -> False


> **记忆要点：读赋值语句，先读右边（创建/取出对象），再读左边（贴标签）。忘掉盒子。**
>
> 用词上，说"把变量绑定到对象"比"把对象赋给变量"更准确（对象在赋值前就存在了）。

# 6.3 同一性、相等性与别名

Lewis Carroll 是 Charles L. Dodgson 教授的笔名：二人**相等**，而且根本就是**同一个人**。用 Python 表达：

In [3]:
# 示例 6.3：charles 与 lewis 引用同一对象（别名）
charles = {'name': 'Charles L. Dodgson', 'born': 1832}
lewis = charles                # lewis 是 charles 的别名
print('lewis is charles:', lewis is charles)
print('id:', id(charles), id(lewis))   # 两个 id 相同
lewis['balance'] = 950         # 通过 lewis 添加一项……
print('charles:', charles)     # ……charles 也跟着变了

lewis is charles: True
id: 1757144734080 1757144734080
charles: {'name': 'Charles L. Dodgson', 'born': 1832, 'balance': 950}


再请一位"冒名顶替者"：他的证件与 Dodgson 教授一模一样，但他**不是**同一个人。

In [4]:
# 示例 6.4：alex 与 charles 相等，但 alex 不是 charles
alex = {'name': 'Charles L. Dodgson', 'born': 1832, 'balance': 950}
print('alex == charles:', alex == charles)      # 值相等
print('alex is not charles:', alex is not charles)  # 但标识不同

alex == charles: True
alex is not charles: True


- **别名（alias）**：两个变量绑定到**同一个**对象（`lewis` / `charles`）；
- **副本**：两个对象**值相等**但**标识不同**（`alex` / `charles`）。

| 概念 | 比较什么 | 运算符 | 速度 |
|---|---|---|---|
| 同一性（identity） | 对象的 id | `is` / `is not` | 快（直接比整数 id，不可重载） |
| 相等性（equality） | 对象的值 | `==` / `!=` | 慢（`a == b` 是 `a.__eq__(b)` 的语法糖，可能很重） |

对象一旦创建，其标识（id）在整个生命周期内**永不变**。`id()` 返回一个表示标识的整数；CPython 中它就是内存地址。`id()` 最常用于**调试**——当两个对象的 `repr` 看起来一样，想弄清它们到底是别名还是副本时。

## 6.3.1 在 == 和 is 之间选择

日常编程更关心**值**，所以 `==` 出现得更多。**用 `is` 的地方，几乎只有一种：拿变量与单例（singleton）比较。**

```python
x is None          # 推荐
x is not None      # 取反的正确写法
```

`None` 是 `is` 最常用的单例；其次是**哨兵（sentinel）对象**：

In [6]:
# 用 is 比较哨兵对象
END_OF_DATA = object()          # 一个独一无二的哨兵

def traverse(node):
    if node is END_OF_DATA:     # 正确：用 is，而不是 ==
        return 'reached end'
    return f'processing {node!r}'

print(traverse('a'))
print(traverse(END_OF_DATA))

processing 'a'
reached end


> **警告：看到除 `is None` 之外的 `is`，多半是错的。** 若不确定就用 `==`——它几乎总能满足需求，也适用于 `None`（只是慢一点）。字符串、整数的"相等"绝不要用 `is` 判断（见 6.7）。

## 6.3.2 元组的相对不可变性

tuple 和其他容器一样，**持有的是对象的引用，而不是对象本身**。所以：

> **tuple 的不可变性，指的是它持有的引用（物理结构）不可变，而不是所引用的对象不可变。**

如果 tuple 里放了一个 list，这个 list 仍然可以就地修改，于是 tuple 的"值"看起来变了。

In [7]:
# 示例 6.5：起初 t1 与 t2 相等；修改 t1 中一个可变项后，二者不等了
t1 = (1, 2, [30, 40])
t2 = (1, 2, [30, 40])
print('t1 == t2:', t1 == t2)         # True
print('id(t1[-1]):', id(t1[-1]))
t1[-1].append(99)                    # 就地修改元组里的可变项
print('t1:', t1)                     # (1, 2, [30, 40, 99])
print('id(t1[-1]):', id(t1[-1]))     # id 不变，只是值变了
print('t1 == t2:', t1 == t2)         # False

t1 == t2: True
id(t1[-1]): 1757144721792
t1: (1, 2, [30, 40, 99])
id(t1[-1]): 1757144721792
t1 == t2: False


> **tuple 里永远不变的是其中各项的标识（id）。** 这既解释了 `+=` 赋值谜题，也是"有些 tuple 不可哈希"的原因（只要包含 list 等不可哈希项，整个 tuple 就不可哈希）。

> 注：`str`、`bytes`、`array.array` 这类**扁平序列**不持有引用，而是把内容直接存在连续内存里，因此没有这个问题。

# 6.4 默认做浅拷贝

复制 list（及其他内置可变容器）最简单的方式是用类型构造函数或全切片：

```python
l2 = list(l1)     # 或 l2 = l1[:]
```

但二者都只做**浅拷贝（shallow copy）**：**只复制最外层容器，副本里填充的是对原容器所持有对象的引用。** 若各项都不可变，没问题；若存在可变项，就可能出现令人不悦的意外。

In [7]:
# 示例 6.6：对包含另一个 list 的 list 做浅拷贝
l1 = [3, [66, 55, 44], (7, 8, 9)]
l2 = list(l1)            # 浅拷贝
l1.append(100)           # 只影响 l1
l1[1].remove(55)         # 影响 l2，因为 l2[1] 与 l1[1] 是同一个 list
print('l1:', l1)
print('l2:', l2)

l2[1] += [33, 22]        # list 的 += 就地修改 → l1[1] 也变
l2[2] += (10, 11)        # tuple 的 += 创建一个新 tuple 并重新绑定 l2[2] → l1[2] 不变
print('l1:', l1)
print('l2:', l2)

l1: [3, [66, 44], (7, 8, 9), 100]
l2: [3, [66, 44], (7, 8, 9)]
l1: [3, [66, 44, 33, 22], (7, 8, 9), 100]
l2: [3, [66, 44, 33, 22], (7, 8, 9, 10, 11)]


> **关键对照：`+=` 对可变对象是"就地修改"，对不可变对象是"创建新对象再重新绑定"。**
>
> `l2[2] += (10, 11)` 等价于 `l2[2] = l2[2] + (10, 11)`——它给 l2 换了个新 tuple，与 l1 无关。

## 6.4.1 任意对象的深拷贝与浅拷贝

`copy` 模块提供两个函数，可对**任意对象**做浅拷贝和深拷贝：

| 函数 | 行为 |
|---|---|
| `copy.copy(x)` | 浅拷贝：副本不共享最外层容器，但仍共享内部的引用 |
| `copy.deepcopy(x)` | 深拷贝：副本连内部对象一起复制，不共享任何可变引用 |

用一辆校车来演示：

In [8]:
# 示例 6.8：校车 Bus（注意构造时对传入的可迭代对象做了复制）
class Bus:
    def __init__(self, passengers=None):
        if passengers is None:
            self.passengers = []
        else:
            self.passengers = list(passengers)

    def pick(self, name):
        self.passengers.append(name)

    def drop(self, name):
        self.passengers.remove(name)

In [9]:
# 示例 6.9：copy 与 deepcopy 产生的不同效果
import copy

bus1 = Bus(['Alice', 'Bill', 'Claire', 'David'])
bus2 = copy.copy(bus1)        # 浅拷贝
bus3 = copy.deepcopy(bus1)    # 深拷贝

print('三个 Bus 的 id:', id(bus1), id(bus2), id(bus3))
bus1.drop('Bill')
print('bus2.passengers:', bus2.passengers)     # 也少了 Bill —— 浅拷贝共享同一列表
print('passengers 的 id:', id(bus1.passengers), id(bus2.passengers), id(bus3.passengers))
print('bus3.passengers:', bus3.passengers)     # 深拷贝，不受影响

三个 Bus 的 id: 2888266365456 2888266353744 2888266487744
bus2.passengers: ['Alice', 'Claire', 'David']
passengers 的 id: 2888266585984 2888266585984 2888266338240
bus3.passengers: ['Alice', 'Bill', 'Claire', 'David']


深拷贝并不简单：对象之间可能存在**循环引用**，简单的递归会陷入死循环。`deepcopy` 会记住已经复制过的对象来处理循环引用。

In [10]:
# 示例 6.10：循环引用，deepcopy 也能正确处理
a = [10, 20]
b = [a, 30]
a.append(b)
print('a =', a)

from copy import deepcopy
c = deepcopy(a)
print('c =', c)

a = [10, 20, [[...], 30]]
c = [10, 20, [[...], 30]]


> 有时深拷贝会"太深"：对象可能引用了不该被复制的**外部资源或单例**。此时可通过实现 `__copy__` 与 `__deepcopy__` 来控制行为（详见 copy 模块文档）。

# 6.5 作为引用的函数参数

Python 唯一的参数传递模式是**共享调用（Call by Sharing）**（JavaScript、Ruby、Java 引用类型也是如此）：

> **函数的每个形参，拿到的是实参所持有引用的副本。也就是说，函数内部的形参是实参的别名。**

推论：
- 函数**可以**修改传入的可变对象（因为它俩指向同一对象）；
- 函数**不能**把传入对象替换成另一个对象（重新绑定只是让局部形参指向别处，对外部无影响）。

In [11]:
# 示例 6.11：函数可以更改它接收的可变对象
def f(a, b):
    a += b
    return a

x, y = 1, 2
print(f(x, y), '→ (x, y) =', (x, y))      # 数字不可变：外部不变

a, b = [1, 2], [3, 4]
print(f(a, b), '→ (a, b) =', (a, b))      # list 的 += 就地修改：外部也变

t, u = (10, 20), (30, 40)
print(f(t, u), '→ (t, u) =', (t, u))      # tuple 的 += 生成新对象：外部不变

3 → (x, y) = (1, 2)
[1, 2, 3, 4] → (a, b) = ([1, 2, 3, 4], [3, 4])
(10, 20, 30, 40) → (t, u) = ((10, 20), (30, 40))


## 6.5.1 不要使用可变类型作为参数的默认值

给可选参数设默认值能保持 API 向后兼容，但**绝不要把可变对象当默认值**。看一个"耍小聪明"的校车：

In [12]:
# 示例 6.12：可变默认值的危险
class HauntedBus:
    '''被幽灵乘客困扰的校车模型'''
    def __init__(self, passengers=[]):   # 危险：默认值是可变的 list
        self.passengers = passengers     # 直接赋别名，不复制

    def pick(self, name):
        self.passengers.append(name)

    def drop(self, name):
        self.passengers.remove(name)

In [13]:
# 示例 6.13：幽灵乘客现身
bus1 = HauntedBus(['Alice', 'Bill'])
print('bus1:', bus1.passengers)
bus1.pick('Charlie'); bus1.drop('Alice')
print('bus1:', bus1.passengers)          # 正常

bus2 = HauntedBus()                      # 用默认 []
bus2.pick('Carrie')
print('bus2:', bus2.passengers)

bus3 = HauntedBus()                      # 也用默认 []
print('bus3:', bus3.passengers)          # ['Carrie'] —— 默认列表已被污染！
bus3.pick('Dave')
print('bus2:', bus2.passengers)          # ['Carrie', 'Dave'] —— Dave 跑到 bus2 里了
print('bus2.passengers is bus3.passengers:', bus2.passengers is bus3.passengers)  # True
print('bus1:', bus1.passengers)          # bus1 是独立列表，不受影响

bus1: ['Alice', 'Bill']
bus1: ['Bill', 'Charlie']
bus2: ['Carrie']
bus3: ['Carrie']
bus2: ['Carrie', 'Dave']
bus2.passengers is bus3.passengers: True
bus1: ['Bill', 'Charlie']


根源是：**默认值在函数定义时（通常加载模块时）求值一次，并成为函数对象的属性。** 若默认值可变且被就地修改，就会影响之后每一次调用。

查看"幽灵学生"的藏身处——函数对象的 `__defaults__` 属性：

In [14]:
# 默认值藏在函数对象的 __defaults__ 里
print('HauntedBus.__init__.__defaults__ =', HauntedBus.__init__.__defaults__)
print('它就是 bus2.passengers:',
      HauntedBus.__init__.__defaults__[0] is bus2.passengers)

HauntedBus.__init__.__defaults__ = (['Carrie', 'Dave'],)
它就是 bus2.passengers: True


> **正解：用 `None` 作默认值，进入函数后再创建新对象。** 这正是 `Bus`（示例 6.8）的做法：

```python
def __init__(self, passengers=None):
    if passengers is None:
        self.passengers = []              # 每次调用都得到新的空列表
    else:
        self.passengers = list(passengers)  # 复制一份，而不是别名
```

## 6.5.2 可变参数的防御性编程

编写接收可变参数的函数时，要**明确调用者是否希望对象被修改**。下面这辆 TwilightBus 就违反了"最少惊讶原则"：

In [15]:
# 示例 6.15：接收可变参数时的风险
class TwilightBus:
    '''让乘客消失的校车'''
    def __init__(self, passengers=None):
        if passengers is None:
            self.passengers = []
        else:
            self.passengers = passengers   # 危险：为实参创建别名

    def pick(self, name):
        self.passengers.append(name)

    def drop(self, name):
        self.passengers.remove(name)

In [16]:
# 示例 6.14：乘客下车后，从原列表里也消失了
basketball_team = ['Sue', 'Tina', 'Maya', 'Diana', 'Pat']
bus = TwilightBus(basketball_team)
bus.drop('Tina')
bus.drop('Pat')
print('basketball_team:', basketball_team)   # ['Sue', 'Maya', 'Diana'] —— 被改坏了

basketball_team: ['Sue', 'Maya', 'Diana']


In [17]:
# 修正：构造时复制一份（防御性编程），并支持任意可迭代对象
class TwilightBus:
    def __init__(self, passengers=None):
        if passengers is None:
            self.passengers = []
        else:
            self.passengers = list(passengers)   # 创建副本，而不是别名

    def pick(self, name):
        self.passengers.append(name)

    def drop(self, name):
        self.passengers.remove(name)

basketball_team = ['Sue', 'Tina', 'Maya', 'Diana', 'Pat']
bus = TwilightBus(basketball_team)
bus.drop('Tina')
print('车内:', bus.passengers)
print('原列表:', basketball_team)    # 不再受影响

车内: ['Sue', 'Maya', 'Diana', 'Pat']
原列表: ['Sue', 'Tina', 'Maya', 'Diana', 'Pat']


> **除非某个方法确实要更改传入的对象，否则把它赋给实例变量之前一定要三思——这会创建别名。** 不确定就复制一份。复制会花一点 CPU 和内存，但在 API 里埋下难以察觉的 bug，代价大得多。

> 附带好处：用 `list(passengers)` 后，参数可以是**任何可迭代对象**（元组、set、数据库查询结果……）。

# 6.6 del 与垃圾回收

> "对象绝不会被显式销毁；但是，当它们变得不可达时，可能会被垃圾回收。" ——《Python 语言参考》

两个令人惊讶的事实：
1. **`del` 是语句，不是函数**——写 `del x`，而不是 `del(x)`（后者能"用"只是因为 `x` 与 `(x)` 同义）。
2. **`del` 删除的是引用，不是对象。** 只有当它删掉的是对象的**最后一个引用**时，对象才可能随之被回收。重新绑定变量也可能让引用计数归零。

In [18]:
# del 删除的是引用，不是对象
a = [1, 2]
b = a
del a            # 只删掉引用 a
print('b =', b)  # [1, 2] 对象仍在，因为 b 还引用它
b = [3]          # b 重新绑定，[1, 2] 的最后一个引用也没了 → 可被回收
print('b =', b)

b = [1, 2]
b = [3]


特殊方法 `__del__` **不负责销毁实例**，也不应在代码里手动调用——它只是给实例一个"释放外部资源"的机会，实际很少需要自己实现。

**CPython 的垃圾回收**：主要靠**引用计数（refcount）**——对象记录有多少引用指向自己，计数归零就**立即**销毁。但引用计数对**循环引用**束手无策，因此 CPython 2.0 增加了**分代垃圾回收**，专门处理互相引用、却整体不可达的对象组。

用 `weakref.finalize` 可以观察对象被回收的时刻：

In [19]:
# 示例 6.16：当引用计数归零时，观察对象生命周期的终结
import weakref

s1 = {1, 2, 3}
s2 = s1                       # s1、s2 是同一对象的别名
def bye():
    print('...like tears in the rain.')

ender = weakref.finalize(s1, bye)   # 注册回调；这是"弱引用"，不增加引用计数
print('alive 1:', ender.alive)      # True
del s1                              # 只删引用 s1，对象还在
print('alive 2:', ender.alive)      # True
s2 = 'spam'                         # 最后一个引用也没了 → 对象被回收，回调触发
print('alive 3:', ender.alive)      # False

alive 1: True
alive 2: True
...like tears in the rain.
alive 3: False


> `finalize` 只持有对 `{1, 2, 3}` 的**弱引用（weak reference）**：弱引用**不会增加**对象的引用计数，因此**不会阻止**对象被回收。弱引用在**缓存**场景中很有用——我们不希望因为缓存里存了引用，导致对象无法被删掉。
>
> （弱引用在第 2 版中被移到了 fluentpython.com 的 "Weak References" 一文。）

# 6.7 Python 对不可变类型施加的把戏

以下是很底层的实现细节，**不保证在其他实现或 CPython 未来版本中成立**，也不必依赖。但因为容易让人误用 `is`，还是值得知道。

**把戏一：对元组做 `t[:]` 或 `tuple(t)`，并不会创建副本，而是返回同一个对象。**

In [20]:
# 示例 6.17：用一个元组构建另一个元组，得到的其实是同一个元组
t1 = (1, 2, 3)
t2 = tuple(t1)
print('tuple(t1) is t1:', t2 is t1)   # True
t3 = t1[:]
print('t1[:] is t1:', t3 is t1)       # True

tuple(t1) is t1: True
t1[:] is t1: True


`str`、`bytes`、`frozenset` 也有同样的行为。注意 `frozenset` 不是序列，没有切片；但它的 `fs.copy()` 同样**返回同一对象**而不是副本——这是个"善意的谎言"，为的是让接口与 `set` 保持一致。

In [21]:
# frozenset.copy() 也返回同一对象
fs = frozenset([1, 2, 3])
fs2 = fs.copy()
print('fs.copy() is fs:', fs2 is fs)   # True

fs.copy() is fs: True


**把戏二：字符串字面量与部分小整数会被"共享"（interning）。**

In [22]:
# 示例 6.18：字符串字面量可能会创建共享的对象
t1 = (1, 2, 3)
t3 = tuple([1, 2, 3])      # 从列表新建，确保是不同对象
print('t3 == t1:', t3 == t1)   # True
print('t3 is t1:', t3 is t1)   # False

s1 = 'ABC'
s2 = 'ABC'
print('s2 is s1:', s2 is s1)   # True —— 字面量被 intern 共享

t3 == t1: True
t3 is t1: False
s2 is s1: True


> 注：在同一个代码对象（如同一个单元格）里写两个相同的**元组字面量**，CPython 可能把它们折叠成同一个常量对象，导致 `is` 为 True。所以上面用 `tuple([1, 2, 3])` 显式新建，才能复现"不同对象"。

共享字符串字面量、缓存常用小整数（0、1、-1 等）是为了省内存、提速，称为 **interning**。但 CPython 并非对所有字符串/整数都这么做，实现细节也没有文档保证。

> **切勿依赖 str 或整数的 interning！比较字符串/整数是否相等用 `==`，不要用 `is`。**

这些把戏（包括 `frozenset.copy()`）只会作用于**不可变类型**，因而**无害**：它们省内存、让解释器更快，不会给你添麻烦。

# 6.8 本章小结

每个 Python 对象都有**标识、类型、值**；只有**值**可能随时间改变。

**关于不可变对象的唯一例外**：不可变**容器**（如元组）若持有可变项的引用，当可变项变化时，容器的"值"实际上也变了。但**容器中各项的标识永远不变**。`frozenset` 没有这个问题——它只能装可哈希元素，而可哈希对象的值定义上永不变。

**"变量保存的是引用"带来的实质影响：**

| 结论 | 说明 |
|---|---|
| 简单赋值不复制 | `b = a` 只是给同一对象贴新标签 |
| `+=` / `*=` 看类型 | 左侧是可变对象时可能**就地修改**；不可变对象则创建新对象 |
| 重新绑定不影响旧对象 | 变量只是改指向新对象；旧对象若失去最后一个引用就被回收 |
| 参数按别名传递 | 即共享调用；函数能改可变实参，除非做本地拷贝或改用不可变对象 |
| 可变默认值很危险 | 参数被就地修改会污染默认值，影响后续每次调用 |

在 CPython 中，引用计数归零对象即被销毁；互相引用、整体不可达的对象组也可被销毁。想"持有引用但不阻止回收"，就用**弱引用**（`WeakValueDictionary`、`WeakSet`、`weakref.finalize` 等的底层基础）。

# 杂谈

## 平等对待所有对象

作者先学的是 Java，一直觉得 Java 的 `==` 是错的：程序员更需要**相等性**，但 Java 的 `==` 比较的是引用（对象），比较字符串还得用 `.equals`，而 `a.equals(b)` 在 a 为 null 时又会抛空指针。

Python 处理得很好：`==` 比较**值**，`is` 比较**引用**；因为有运算符重载，`==` 能正确处理标准库中的所有对象，**包括 `None`**（它是普通对象，不像 Java 的 null）。

自定义类可自行实现 `__eq__` 来决定比较方式；若不覆盖，则继承自 `object` 的 `__eq__` 比较 id，所以用户定义类的实例默认都"不相等"。

## 可变性

若所有对象都不可变，本章就没必要存在了——此时"变量直接持对象"还是"持引用"没有区别（`a == b` 为真且二者不可变时，它们可能就是同一对象，这正是字符串可以 interning 的原因）。**只有当对象可变（如 list）时，标识才重要。**

- "纯"函数式编程中所有数据不可变，追加操作会返回新容器（Elixir 的所有内置类型都不可变）；
- Python 不是函数式语言，用户定义类的实例**默认可变**。想自己造不可变对象要格外小心：**每个属性也必须不可变**，否则会重演元组的尴尬；
- 可变对象也是多线程编程难以成功的主因：改完不同步会出错，过度同步又会死锁。Erlang/Elixir 面向高可用电信场景，自然默认选不可变数据。

## 对象销毁与垃圾回收

Python 没有"直接销毁对象"的机制，这其实是好事：若对象可被随时销毁，那些指向它的引用怎么办？

CPython 主要靠**引用计数**（易实现，但遇到引用循环会泄漏），2.0 起加入**分代垃圾回收器**来处理循环引用。引用计数仍是基准：归零即立刻销毁。因此在 CPython 中下面写法是安全的：

```python
open('test.txt', 'wt', encoding='utf-8').write('1, 2, 3')
```

因为 `write` 返回后文件对象引用计数归零，会被立刻关闭并回收。但在 **Jython / IronPython** 中不安全（它们用宿主 JVM/.NET 的 GC，不依赖引用计数）。**无论哪种实现，最可靠的做法都是用 `with` 显式管理资源：**

```python
with open('test.txt', 'wt', encoding='utf-8') as fp:
    fp.write('1, 2, 3')
```

## 参数传递：共享调用（Call by Sharing）

"参数按值传递，只不过这里的值是引用"——这么说没错，但容易引起误解。更准确的说法是：

> **函数拿到的是实参所持引用的副本。**

因此：若引用的是可变对象，对象的值可能被修改，但标识不变；又因为拿到的是**副本**，函数体内**重新绑定**这个引用，对外部毫无影响。作者参考 Michael L. Scott《Programming Language Pragmatics》8.3.1 节的术语，采用了"共享调用（Call by Sharing）"这一说法。

> 对照：（1）按值传递——函数拿到实参的副本；（2）按引用传递——函数拿到实参的指针。

# 6.9 延伸阅读

- 《Python 语言参考 · Data model》开头——对象标识与值的权威解释；
- Wesley Chun，EuroPython 2011《Understanding Python's Memory Model, Mutability, and Methods》；
- Doug Hellmann，《copy — Duplicate Objects》与《weakref — Impermanent References to Objects》；
- CPython `gc` 模块文档 + Pablo Galindo《垃圾回收器设计》（Python 开发者指南）；
- PEP 442 – Safe object finalization（CPython 3.4 改进了 `__del__` 的处理）；
- 维基百科：String interning、Haddock's Eyes（本章开头的歌）。